In [ ]:
# Single cell — run everything at once

from google.colab import drive
drive.mount('/content/drive')

!pip -q install transformers accelerate sentencepiece rank_bm25 peft

import re, json, gc, time, torch
from pathlib import Path
from collections import Counter, defaultdict
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

PROJECT_DIR = Path("/content/drive/MyDrive/421-Project")

# Load data
with open(PROJECT_DIR / "hotpot_dev_distractor_v1.json") as f:
    hotpot_dev = json.load(f)
with open(PROJECT_DIR / "hotpot_train_v1.1.json") as f:
    hotpot_train = json.load(f)

print(f"Dev: {len(hotpot_dev)} | Train: {len(hotpot_train)}")

# Helper functions
def normalize_text(s):
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9\s]", "", s)
    return " ".join(s.split())

def compute_em(pred, gold):
    return int(normalize_text(pred) == normalize_text(gold))

def compute_f1(pred, gold):
    pred_tokens = normalize_text(pred).split()
    gold_tokens = normalize_text(gold).split()
    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if not pred_tokens or not gold_tokens:
        return int(pred_tokens == gold_tokens)
    if num_same == 0:
        return 0.0
    p = num_same / len(pred_tokens)
    r = num_same / len(gold_tokens)
    return 2 * p * r / (p + r)

def simple_tokenize(text):
    return re.findall(r"\w+", text.lower())

def get_qtype(ex):
    return ex.get("type", "unknown")

def hotpot_context_to_text(c):
    title, sents = c
    return f"Title: {title}\n{' '.join(sents)}"

def build_hotpot_all_context(ex):
    return "\n\n".join(hotpot_context_to_text(c) for c in ex["context"])

def build_hotpot_supporting_context(ex):
    supp = set(t for t, _ in ex["supporting_facts"])
    return "\n\n".join(hotpot_context_to_text(c) for c in ex["context"] if c[0] in supp)

def build_hotpot_bm25_topk(ex, k=5):
    para_texts = [hotpot_context_to_text(c) for c in ex["context"]]
    bm25 = BM25Okapi([simple_tokenize(t) for t in para_texts])
    scores = bm25.get_scores(simple_tokenize(ex["question"]))
    ranked = sorted(zip(ex["context"], para_texts, scores), key=lambda x: x[2], reverse=True)[:k]
    context = "\n\n".join(t for _, t, _ in ranked)
    supp = set(t for t, _ in ex["supporting_facts"])
    retrieved = set(c[0] for c, _, _ in ranked)
    recall = len(supp & retrieved) / len(supp) if supp else 0
    return context, recall, int(supp <= retrieved)

def build_qa_prompt(question, context):
    return f"Answer the question using the context below.\n\nQuestion: {question}\n\nContext:\n{context}\n\nAnswer:"

def build_cot_prompt_v3(question, context):
    return f'Answer the question using the context below.\n\nReason step by step, then on a new line write only "Answer:" followed by the short final answer.\n\nQuestion: {question}\n\nContext:\n{context}\n\nReasoning:'

def extract_answer_v3(output):
    matches = list(re.finditer(r"(?:Final\s+)?Answer:\s*(.*)", output, re.IGNORECASE))
    if matches:
        return matches[-1].group(1).strip().rstrip(".")
    match = re.search(r"(?:so,?\s+)?the\s+(?:final\s+)?answer\s+is\s+(.*)", output, re.IGNORECASE)
    if match:
        return match.group(1).strip().rstrip(".")
    first_sent = re.split(r'[.\n]', output.strip())[0].strip()
    return first_sent if first_sent else output.strip()

# Load model
device = "cuda"
MODEL_NAME = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto")

def run_model(prompt, max_tokens=32):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_tokens, num_beams=4,
                                  early_stopping=True, no_repeat_ngram_size=3 if max_tokens > 32 else 0)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

print(f"\nLoaded: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.0f}M")
print(f"Memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print("Ready.")

Mounted at /content/drive
Dev: 7405 | Train: 90447


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


Loaded: google/flan-t5-large
Parameters: 783M
Memory: 1.9 GB
Ready.


In [ ]:
# HotpotQA Large: B1, B2, B3 in one pass

import time

total = len(hotpot_dev)
results = {
    "B1": {"em": defaultdict(list), "f1": defaultdict(list)},
    "B2": {"em": defaultdict(list), "f1": defaultdict(list), "recall": [], "full_recall": 0},
    "B3": {"em": defaultdict(list), "f1": defaultdict(list)},
}

start = time.time()

for i, ex in enumerate(hotpot_dev, start=1):
    qtype = get_qtype(ex)

    # B1: All paragraphs
    pred = run_model(build_qa_prompt(ex["question"], build_hotpot_all_context(ex)))
    results["B1"]["em"][qtype].append(compute_em(pred, ex["answer"]))
    results["B1"]["f1"][qtype].append(compute_f1(pred, ex["answer"]))

    # B2: BM25 top-5
    bm25_ctx, recall, full = build_hotpot_bm25_topk(ex, k=5)
    pred = run_model(build_qa_prompt(ex["question"], bm25_ctx))
    results["B2"]["em"][qtype].append(compute_em(pred, ex["answer"]))
    results["B2"]["f1"][qtype].append(compute_f1(pred, ex["answer"]))
    results["B2"]["recall"].append(recall)
    results["B2"]["full_recall"] += full

    # B3: Oracle
    pred = run_model(build_qa_prompt(ex["question"], build_hotpot_supporting_context(ex)))
    results["B3"]["em"][qtype].append(compute_em(pred, ex["answer"]))
    results["B3"]["f1"][qtype].append(compute_f1(pred, ex["answer"]))

    if i % 500 == 0:
        elapsed = time.time() - start
        for name in ["B1", "B2", "B3"]:
            all_em = [s for q in results[name]["em"] for s in results[name]["em"][q]]
            print(f"  {name} EM={sum(all_em)/len(all_em):.3f}", end="")
        print(f"  Time={elapsed:.0f}s [{i}/{total}]")

# Print results
print("\n" + "=" * 80)
print(f"{'Baseline':<20} {'Bridge EM':>12} {'Comp EM':>12} {'All EM':>12} {'All F1':>12}")
print("=" * 80)

for name in ["B1", "B2", "B3"]:
    r = results[name]
    all_em = [s for q in r["em"] for s in r["em"][q]]
    all_f1 = [s for q in r["f1"] for s in r["f1"][q]]
    bridge_em = sum(r["em"]["bridge"]) / len(r["em"]["bridge"])
    comp_em = sum(r["em"]["comparison"]) / len(r["em"]["comparison"])
    print(f"{name + ': Large (ZS)':<20} {bridge_em:>12.3f} {comp_em:>12.3f} "
          f"{sum(all_em)/len(all_em):>12.3f} {sum(all_f1)/len(all_f1):>12.3f}")

print("=" * 80)
print(f"\nB2 Retrieval Recall@5: {sum(results['B2']['recall'])/len(results['B2']['recall']):.3f} (partial)")
print(f"B2 Full Recall@5: {results['B2']['full_recall']/total:.3f} ({results['B2']['full_recall']}/{total})")

  B1 EM=0.314  B2 EM=0.262  B3 EM=0.446  Time=487s [500/7405]
  B1 EM=0.331  B2 EM=0.286  B3 EM=0.455  Time=979s [1000/7405]
  B1 EM=0.324  B2 EM=0.291  B3 EM=0.463  Time=1480s [1500/7405]
  B1 EM=0.319  B2 EM=0.288  B3 EM=0.458  Time=1975s [2000/7405]
  B1 EM=0.316  B2 EM=0.286  B3 EM=0.458  Time=2472s [2500/7405]
  B1 EM=0.318  B2 EM=0.290  B3 EM=0.458  Time=2976s [3000/7405]
  B1 EM=0.320  B2 EM=0.291  B3 EM=0.456  Time=3468s [3500/7405]
  B1 EM=0.322  B2 EM=0.289  B3 EM=0.457  Time=3961s [4000/7405]
  B1 EM=0.318  B2 EM=0.284  B3 EM=0.455  Time=4447s [4500/7405]
  B1 EM=0.320  B2 EM=0.284  B3 EM=0.455  Time=4934s [5000/7405]
  B1 EM=0.317  B2 EM=0.282  B3 EM=0.451  Time=5425s [5500/7405]
  B1 EM=0.318  B2 EM=0.281  B3 EM=0.452  Time=5921s [6000/7405]
  B1 EM=0.318  B2 EM=0.283  B3 EM=0.454  Time=6407s [6500/7405]
  B1 EM=0.316  B2 EM=0.282  B3 EM=0.452  Time=6881s [7000/7405]

Baseline                Bridge EM      Comp EM       All EM       All F1
B1: Large (ZS)              0.287

In [ ]:
# HotpotQA Large: CoT zero-shot (oracle context)

import time

total = len(hotpot_dev)
em_scores_cot0 = defaultdict(list)
f1_scores_cot0 = defaultdict(list)

start = time.time()

for i, ex in enumerate(hotpot_dev, start=1):
    qtype = get_qtype(ex)
    context = build_hotpot_supporting_context(ex)
    prompt = build_cot_prompt_v3(ex["question"], context)
    raw = run_model(prompt, max_tokens=256)
    pred = extract_answer_v3(raw)

    em_scores_cot0[qtype].append(compute_em(pred, ex["answer"]))
    f1_scores_cot0[qtype].append(compute_f1(pred, ex["answer"]))

    if i % 500 == 0:
        all_em = [s for q in em_scores_cot0 for s in em_scores_cot0[q]]
        all_f1 = [s for q in f1_scores_cot0 for s in f1_scores_cot0[q]]
        elapsed = time.time() - start
        print(f"[{i}/{total}] EM={sum(all_em)/len(all_em):.3f} "
              f"F1={sum(all_f1)/len(all_f1):.3f} Time={elapsed:.0f}s")

print("\n" + "=" * 80)
print(f"{'CoT 0-shot oracle':<25} {'Bridge EM':>12} {'Comp EM':>12} {'All EM':>12} {'All F1':>12}")
print("=" * 80)

all_em = [s for q in em_scores_cot0 for s in em_scores_cot0[q]]
all_f1 = [s for q in f1_scores_cot0 for s in f1_scores_cot0[q]]
bridge_em = sum(em_scores_cot0["bridge"]) / len(em_scores_cot0["bridge"])
comp_em = sum(em_scores_cot0["comparison"]) / len(em_scores_cot0["comparison"])

print(f"{'CoT 0-shot oracle':<25} {bridge_em:>12.3f} {comp_em:>12.3f} "
      f"{sum(all_em)/len(all_em):>12.3f} {sum(all_f1)/len(all_f1):>12.3f}")
print("=" * 80)

print(f"\nComparison — B3 Oracle Large (ZS): EM=0.453, F1=0.644")
print(f"Comparison — CoT 0-shot XXL:       EM=0.586, F1=0.754")

[500/7405] EM=0.458 F1=0.634 Time=188s
[1000/7405] EM=0.464 F1=0.648 Time=374s
[1500/7405] EM=0.456 F1=0.643 Time=566s
[2000/7405] EM=0.446 F1=0.631 Time=756s
[2500/7405] EM=0.449 F1=0.633 Time=952s
[3000/7405] EM=0.449 F1=0.633 Time=1145s
[3500/7405] EM=0.450 F1=0.633 Time=1333s
[4000/7405] EM=0.449 F1=0.636 Time=1528s
[4500/7405] EM=0.449 F1=0.636 Time=1716s
[5000/7405] EM=0.448 F1=0.634 Time=1906s
[5500/7405] EM=0.443 F1=0.633 Time=2103s
[6000/7405] EM=0.442 F1=0.632 Time=2294s
[6500/7405] EM=0.441 F1=0.631 Time=2484s
[7000/7405] EM=0.440 F1=0.631 Time=2675s

CoT 0-shot oracle            Bridge EM      Comp EM       All EM       All F1
CoT 0-shot oracle                0.424        0.506        0.440        0.630

Comparison — B3 Oracle Large (ZS): EM=0.453, F1=0.644
Comparison — CoT 0-shot XXL:       EM=0.586, F1=0.754


In [ ]:
# HotpotQA Method 5: LoRA fine-tuning setup

from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset, DataLoader
import random

# LoRA config
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q", "v"],
)

model.enable_input_require_grads()
model.gradient_checkpointing_enable()
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

print(f"Memory after LoRA: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# Training dataset
class HotpotMultitaskDataset(Dataset):
    def __init__(self, data, tok, max_input_len=512, max_target_len=64, max_examples=5000):
        self.tok = tok
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len
        self.examples = []

        random.seed(42)
        sample = random.sample(data, min(max_examples, len(data)))

        for ex in sample:
            question = ex["question"]
            answer = ex["answer"]
            supporting_titles = set(t for t, _ in ex["supporting_facts"])
            supporting_paras = [c for c in ex["context"] if c[0] in supporting_titles]
            context = "\n\n".join(hotpot_context_to_text(c) for c in supporting_paras)
            titles = [c[0] for c in supporting_paras]

            input_text = (f"Answer the question and identify which paragraphs support your answer.\n\n"
                         f"Question: {question}\n\nContext:\n{context}\n\n"
                         f"Answer and supporting paragraphs:")
            target_text = f"Answer: {answer}\nSupporting: {', '.join(titles)}"
            self.examples.append((input_text, target_text))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        input_text, target_text = self.examples[idx]
        inputs = self.tok(input_text, max_length=self.max_input_len,
                         truncation=True, padding="max_length", return_tensors="pt")
        targets = self.tok(target_text, max_length=self.max_target_len,
                          truncation=True, padding="max_length", return_tensors="pt")
        labels = targets["input_ids"].squeeze()
        labels[labels == self.tok.pad_token_id] = -100
        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels": labels,
        }

train_dataset = HotpotMultitaskDataset(hotpot_train, tokenizer, max_examples=5000)
print(f"\nTraining examples: {len(train_dataset)}")

# Verify one example
inp, tgt = train_dataset.examples[0]
print(f"\nSample input (first 300 chars):\n{inp[:300]}")
print(f"\nSample target:\n{tgt}")

trainable params: 4,718,592 || all params: 787,868,672 || trainable%: 0.5989
Memory after LoRA: 1.9 GB

Training examples: 5000

Sample input (first 300 chars):
Answer the question and identify which paragraphs support your answer.

Question: Which Emmett's Mark actor also played in the HBO series "The Wire"?

Context:
Title: Emmett's Mark
Emmett's Mark is a 2002 American thriller film directed by Keith Snyder and starring Scott Wolf, Khandi Alexander, Tali

Sample target:
Answer: John Doman
Supporting: Emmett's Mark, John Doman


In [ ]:
# HotpotQA Method 5: LoRA training

import time

BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8
EPOCHS = 3
LR = 5e-5

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
optimizer = torch.optim.AdamW(peft_model.parameters(), lr=LR, weight_decay=0.01, eps=1e-4)

total_steps = len(train_loader) * EPOCHS // GRAD_ACCUM_STEPS
warmup_steps = total_steps // 10

def get_lr(step):
    if step < warmup_steps:
        return max(0.01, step / warmup_steps)
    return max(0.1, 1.0 - (step - warmup_steps) / (total_steps - warmup_steps))

peft_model.train()
global_step = 0
best_loss = float("inf")

print(f"Training config:")
print(f"  Examples: {len(train_dataset)}")
print(f"  Effective batch: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"  Total steps: {total_steps}")
print(f"  LR: {LR}")
print()

for epoch in range(EPOCHS):
    epoch_loss = 0
    epoch_steps = 0
    nan_count = 0
    start = time.time()
    optimizer.zero_grad()

    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        try:
            outputs = peft_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            loss = outputs.loss / GRAD_ACCUM_STEPS

            if torch.isnan(loss):
                nan_count += 1
                optimizer.zero_grad()
                continue

            loss.backward()
            epoch_loss += outputs.loss.item()
            epoch_steps += 1

        except RuntimeError as e:
            if "out of memory" in str(e):
                torch.cuda.empty_cache()
                optimizer.zero_grad()
                nan_count += 1
                continue
            raise

        if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0:
            lr_scale = get_lr(global_step)
            for pg in optimizer.param_groups:
                pg["lr"] = LR * lr_scale
            torch.nn.utils.clip_grad_norm_(peft_model.parameters(), 0.5)
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1

        if (batch_idx + 1) % 500 == 0:
            avg_loss = epoch_loss / max(epoch_steps, 1)
            elapsed = time.time() - start
            mem = torch.cuda.memory_allocated() / 1e9
            print(f"  Epoch {epoch+1} [{batch_idx+1}/{len(train_loader)}] "
                  f"loss={avg_loss:.4f} nan={nan_count} "
                  f"mem={mem:.1f}GB time={elapsed:.0f}s")

    avg_loss = epoch_loss / max(epoch_steps, 1)
    elapsed = time.time() - start
    print(f"\nEpoch {epoch+1}/{EPOCHS} — loss: {avg_loss:.4f} "
          f"nan: {nan_count} time: {elapsed:.0f}s")

    if avg_loss < best_loss and avg_loss > 0:
        best_loss = avg_loss
        peft_model.save_pretrained(
            "/content/drive/MyDrive/421-Project/hotpot_lora_method5_best"
        )
        print(f"  Saved (loss={best_loss:.4f})")
    print()

print(f"Training complete! Best loss: {best_loss:.4f}")

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Training config:
  Examples: 5000
  Effective batch: 16
  Total steps: 937
  LR: 5e-05

  Epoch 1 [500/2500] loss=9.7831 nan=0 mem=2.0GB time=320s
  Epoch 1 [1000/2500] loss=9.7559 nan=0 mem=1.9GB time=640s
  Epoch 1 [1500/2500] loss=9.6625 nan=0 mem=2.0GB time=961s
  Epoch 1 [2000/2500] loss=9.4896 nan=0 mem=1.9GB time=1281s
  Epoch 1 [2500/2500] loss=9.3411 nan=0 mem=2.0GB time=1600s

Epoch 1/3 — loss: 9.3411 nan: 0 time: 1600s
  Saved (loss=9.3411)

  Epoch 2 [500/2500] loss=8.5896 nan=0 mem=2.0GB time=321s
  Epoch 2 [1000/2500] loss=8.5156 nan=0 mem=1.9GB time=641s
  Epoch 2 [1500/2500] loss=8.4580 nan=0 mem=2.0GB time=963s
  Epoch 2 [2000/2500] loss=8.4057 nan=0 mem=1.9GB time=1284s
  Epoch 2 [2500/2500] loss=8.3618 nan=0 mem=2.0GB time=1606s

Epoch 2/3 — loss: 8.3618 nan: 0 time: 1606s
  Saved (loss=8.3618)

  Epoch 3 [500/2500] loss=8.1409 nan=0 mem=2.0GB time=320s
  Epoch 3 [1000/2500] loss=8.1265 nan=0 mem=1.9GB time=642s
  Epoch 3 [1500/2500] loss=8.1085 nan=0 mem=2.0GB time=

In [ ]:
# Define M5 prompt and answer extraction functions

def build_m5_prompt(question, context):
    return (f"Answer the question and identify which paragraphs support your answer.\n\n"
            f"Question: {question}\n\nContext:\n{context}\n\n"
            f"Answer and supporting paragraphs:")

def extract_m5_answer(output):
    import re
    match = re.search(r"Answer:\s*(.*?)(?:\n|Supporting:|$)", output, re.IGNORECASE)
    if match:
        return match.group(1).strip().rstrip(".")
    lines = [l.strip() for l in output.strip().split('\n') if l.strip()]
    return lines[0] if lines else output.strip()

print("M5 prompt helpers defined.")

M5 prompt helpers defined.


In [ ]:
# M5 Evaluation — optimized (single pass, shorter generation)

import time

total = len(hotpot_dev)
results_m5 = {
    "oracle": {"em": defaultdict(list), "f1": defaultdict(list)},
    "bm25":   {"em": defaultdict(list), "f1": defaultdict(list)},
}

def run_peft_model_fast(prompt, max_tokens=32):
    """Shorter max_tokens since we only need the answer line."""
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=512
    ).to(device)
    with torch.no_grad():
        outputs = peft_model.generate(
            **inputs, max_new_tokens=max_tokens,
            num_beams=2,        # reduced from 4
            early_stopping=True
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

start = time.time()

for i, ex in enumerate(hotpot_dev, start=1):
    qtype = get_qtype(ex)
    q = ex["question"]

    # Oracle
    oracle_ctx = build_hotpot_supporting_context(ex)
    raw = run_peft_model_fast(build_m5_prompt(q, oracle_ctx))
    pred = extract_m5_answer(raw)
    results_m5["oracle"]["em"][qtype].append(compute_em(pred, ex["answer"]))
    results_m5["oracle"]["f1"][qtype].append(compute_f1(pred, ex["answer"]))

    # BM25
    bm25_ctx, _, _ = build_hotpot_bm25_topk(ex, k=5)
    raw = run_peft_model_fast(build_m5_prompt(q, bm25_ctx))
    pred = extract_m5_answer(raw)
    results_m5["bm25"]["em"][qtype].append(compute_em(pred, ex["answer"]))
    results_m5["bm25"]["f1"][qtype].append(compute_f1(pred, ex["answer"]))

    if i % 500 == 0:
        elapsed = time.time() - start
        est_total = elapsed / i * total
        for name in ["oracle", "bm25"]:
            all_em = [s for q in results_m5[name]["em"] for s in results_m5[name]["em"][q]]
            print(f"  M5-{name} EM={sum(all_em)/len(all_em):.3f}", end="")
        print(f"  [{i}/{total}] {elapsed:.0f}s elapsed, ~{(est_total-elapsed)/3600:.1f}h remaining")

print("\n" + "=" * 80)
print(f"{'Method':<25} {'Bridge EM':>12} {'Comp EM':>12} {'All EM':>12} {'All F1':>12}")
print("=" * 80)

for name, label in [("oracle", "M5 LoRA oracle"), ("bm25", "M5 LoRA BM25")]:
    r = results_m5[name]
    all_em = [s for q in r["em"] for s in r["em"][q]]
    all_f1 = [s for q in r["f1"] for s in r["f1"][q]]
    bridge_em = sum(r["em"]["bridge"]) / len(r["em"]["bridge"])
    comp_em = sum(r["em"]["comparison"]) / len(r["em"]["comparison"])
    print(f"{label + ' (Large)':<25} {bridge_em:>12.3f} {comp_em:>12.3f} "
          f"{sum(all_em)/len(all_em):>12.3f} {sum(all_f1)/len(all_f1):>12.3f}")

print("=" * 80)
print(f"\nComparison — B3 Oracle Large (ZS): EM=0.453")
print(f"Comparison — B3 Oracle XXL (ZS):   EM=0.548")

  M5-oracle EM=0.000  M5-bm25 EM=0.000  [500/7405] 451s elapsed, ~1.7h remaining


KeyboardInterrupt: 

In [ ]:
# Diagnose M5 output format

peft_model.eval()

for i, ex in enumerate(hotpot_dev[:5], start=1):
    oracle_ctx = build_hotpot_supporting_context(ex)
    prompt = build_m5_prompt(ex["question"], oracle_ctx)

    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=512
    ).to(device)
    with torch.no_grad():
        outputs = peft_model.generate(
            **inputs, max_new_tokens=32, num_beams=2, early_stopping=True
        )
    raw = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    pred = extract_m5_answer(raw)

    print(f"\n{'='*60}")
    print(f"Q: {ex['question'][:70]}")
    print(f"Gold: {ex['answer']}")
    print(f"Raw output: '{raw}'")
    print(f"Extracted: '{pred}'")
    print(f"EM: {compute_em(pred, ex['answer'])}")


Q: Were Scott Derrickson and Ed Wood of the same nationality?
Gold: yes
Raw output: 'Answer: American Supporting: Scott Derrickson, Ed Wood'
Extracted: 'American'
EM: 0

Q: What government position was held by the woman who portrayed Corliss A
Gold: Chief of Protocol
Raw output: 'Answer: United States ambassador'
Extracted: 'United States ambassador'
EM: 0

Q: What science fantasy young adult series, told in first person, has a s
Gold: Animorphs
Raw output: 'Answer: Animorphs'
Extracted: 'Animorphs'
EM: 1

Q: Are the Laleli Mosque and Esma Sultan Mansion located in the same neig
Gold: no
Raw output: 'Answer: Ortaköy Supporting: Laleli Mosque, Esma Sultan Mansion'
Extracted: 'Ortaköy'
EM: 0

Q: The director of the romantic comedy "Big Stone Gap" is based in what N
Gold: Greenwich Village, New York City
Raw output: 'Answer: Greenwich Village Supporting: Adriana Trigiani, Big Stone Gap (film)'
Extracted: 'Greenwich Village'
EM: 0


In [ ]:
# Check yes/no distribution in training sample
yn_train = sum(1 for inp, tgt in train_dataset.examples
               if tgt.startswith("Answer: yes") or tgt.startswith("Answer: no"))
print(f"Yes/No in training sample: {yn_train}/5000 ({yn_train/50:.1f}%)")

# Also check what the model outputs for yes/no questions specifically
yn_examples = [(ex, build_hotpot_supporting_context(ex))
               for ex in hotpot_dev[:100]
               if ex["answer"].lower() in ["yes", "no"]][:5]

print(f"\nYes/No examples in first 100 dev: {len(yn_examples)}")
for ex, ctx in yn_examples:
    inputs = tokenizer(build_m5_prompt(ex["question"], ctx),
                      return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        out = peft_model.generate(**inputs, max_new_tokens=32, num_beams=2, early_stopping=True)
    raw = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"\nQ: {ex['question'][:70]}")
    print(f"Gold: {ex['answer']} | Raw: {raw[:80]}")

Yes/No in training sample: 318/5000 (6.4%)

Yes/No examples in first 100 dev: 5

Q: Were Scott Derrickson and Ed Wood of the same nationality?
Gold: yes | Raw: Answer: American Supporting: Scott Derrickson, Ed Wood

Q: Are the Laleli Mosque and Esma Sultan Mansion located in the same neig
Gold: no | Raw: Answer: Ortaköy Supporting: Laleli Mosque, Esma Sultan Mansion

Q: Are Local H and For Against both from the United States?
Gold: yes | Raw: Answer: Yes, Local H Supporting: Local H, For Against

Q: Are Random House Tower and 888 7th Avenue both used for real estate?
Gold: no | Raw: Answer: Random House Tower Supporting

Q: Are Giuseppe Verdi and Ambroise Thomas both Opera composers ?
Gold: yes | Raw: Answer: Giuseppe Verdi, Ambroise Thomas Supporting: Giuseppe Verdi, Ambroise Tho


In [ ]:
# Fix extractor + quick 200-example eval

import re

def extract_m5_answer_v2(output):
    """Improved extractor with yes/no handling."""
    match = re.search(r"Answer:\s*(.*?)(?:\n|Supporting:|$)", output, re.IGNORECASE)
    if match:
        ans = match.group(1).strip().rstrip(".")
        # Yes/no fix: if answer contains yes or no, return just that
        ans_lower = ans.lower()
        if ans_lower.startswith("yes"):
            return "yes"
        if ans_lower.startswith("no"):
            return "no"
        return ans
    lines = [l.strip() for l in output.strip().split('\n') if l.strip()]
    return lines[0] if lines else output.strip()

# Quick eval on 200 examples
sample = hotpot_dev[:200]
em_oracle, f1_oracle = [], []
em_bm25, f1_bm25 = [], []

for ex in sample:
    q = ex["question"]

    # Oracle
    oracle_ctx = build_hotpot_supporting_context(ex)
    inputs = tokenizer(build_m5_prompt(q, oracle_ctx),
                      return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        out = peft_model.generate(**inputs, max_new_tokens=32, num_beams=2, early_stopping=True)
    pred = extract_m5_answer_v2(tokenizer.decode(out[0], skip_special_tokens=True))
    em_oracle.append(compute_em(pred, ex["answer"]))
    f1_oracle.append(compute_f1(pred, ex["answer"]))

    # BM25
    bm25_ctx, _, _ = build_hotpot_bm25_topk(ex, k=5)
    inputs = tokenizer(build_m5_prompt(q, bm25_ctx),
                      return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        out = peft_model.generate(**inputs, max_new_tokens=32, num_beams=2, early_stopping=True)
    pred = extract_m5_answer_v2(tokenizer.decode(out[0], skip_special_tokens=True))
    em_bm25.append(compute_em(pred, ex["answer"]))
    f1_bm25.append(compute_f1(pred, ex["answer"]))

print(f"Quick eval on {len(sample)} examples:")
print(f"  M5 LoRA oracle: EM={sum(em_oracle)/len(em_oracle):.3f} F1={sum(f1_oracle)/len(f1_oracle):.3f}")
print(f"  M5 LoRA BM25:   EM={sum(em_bm25)/len(em_bm25):.3f} F1={sum(f1_bm25)/len(f1_bm25):.3f}")
print(f"\nComparison — B3 Oracle Large (ZS): EM=0.453")

Quick eval on 200 examples:
  M5 LoRA oracle: EM=0.425 F1=0.611
  M5 LoRA BM25:   EM=0.070 F1=0.229

Comparison — B3 Oracle Large (ZS): EM=0.453


In [ ]:
# Proper cleanup — remove LoRA and reload base model fresh

import gc
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Delete everything
if 'peft_model' in dir():
    del peft_model
if 'model' in dir():
    del model
if 'optimizer' in dir():
    del optimizer
gc.collect()
torch.cuda.empty_cache()

print(f"After cleanup: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# Reload base model fresh
MODEL_NAME = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
device = "cuda"
print(f"Base model reloaded: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# Now apply LoRA cleanly
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q", "v"],
)
model.enable_input_require_grads()
model.gradient_checkpointing_enable()
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()
print(f"LoRA applied cleanly: {torch.cuda.memory_allocated()/1e9:.1f} GB")

After cleanup: 1.9 GB


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Base model reloaded: 3.8 GB
trainable params: 4,718,592 || all params: 787,868,672 || trainable%: 0.5989
LoRA applied cleanly: 3.8 GB


In [ ]:
# M5 v2 Training — run after clean LoRA setup

import time
from torch.utils.data import DataLoader

BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
EPOCHS = 3
LR = 5e-5

train_loader = DataLoader(train_dataset_v2, batch_size=BATCH_SIZE, shuffle=True)
optimizer = torch.optim.AdamW(peft_model.parameters(), lr=LR, weight_decay=0.01, eps=1e-4)

total_steps = len(train_loader) * EPOCHS // GRAD_ACCUM_STEPS
warmup_steps = total_steps // 10

def get_lr(step):
    if step < warmup_steps:
        return max(0.01, step / warmup_steps)
    return max(0.1, 1.0 - (step - warmup_steps) / (total_steps - warmup_steps))

peft_model.train()
global_step = 0
best_loss = float("inf")

print(f"Training config v2:")
print(f"  Examples: {len(train_dataset_v2)} (oracle + BM25 mixed)")
print(f"  Effective batch: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"  Total steps: {total_steps}")
print()

for epoch in range(EPOCHS):
    epoch_loss = 0
    epoch_steps = 0
    nan_count = 0
    start = time.time()
    optimizer.zero_grad()

    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        try:
            outputs = peft_model(input_ids=input_ids,
                                attention_mask=attention_mask,
                                labels=labels)
            loss = outputs.loss / GRAD_ACCUM_STEPS

            if torch.isnan(loss):
                nan_count += 1
                optimizer.zero_grad()
                continue

            loss.backward()
            epoch_loss += outputs.loss.item()
            epoch_steps += 1

        except RuntimeError as e:
            if "out of memory" in str(e):
                torch.cuda.empty_cache()
                optimizer.zero_grad()
                nan_count += 1
                continue
            raise

        if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0:
            lr_scale = get_lr(global_step)
            for pg in optimizer.param_groups:
                pg["lr"] = LR * lr_scale
            torch.nn.utils.clip_grad_norm_(peft_model.parameters(), 0.5)
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1

        if (batch_idx + 1) % 500 == 0:
            avg_loss = epoch_loss / max(epoch_steps, 1)
            elapsed = time.time() - start
            print(f"  Epoch {epoch+1} [{batch_idx+1}/{len(train_loader)}] "
                  f"loss={avg_loss:.4f} nan={nan_count} time={elapsed:.0f}s")

    avg_loss = epoch_loss / max(epoch_steps, 1)
    elapsed = time.time() - start
    print(f"\nEpoch {epoch+1}/{EPOCHS} — loss: {avg_loss:.4f} nan: {nan_count} time: {elapsed:.0f}s")

    if avg_loss < best_loss and avg_loss > 0:
        best_loss = avg_loss
        peft_model.save_pretrained(
            "/content/drive/MyDrive/421-Project/hotpot_lora_v2_best"
        )
        print(f"  Saved (loss={best_loss:.4f})")
    print()

print(f"Training complete! Best loss: {best_loss:.4f}")

Training config v2:
  Examples: 16000 (oracle + BM25 mixed)
  Effective batch: 16
  Total steps: 3000

  Epoch 1 [500/4000] loss=9.6086 nan=0 time=327s
  Epoch 1 [1000/4000] loss=9.5708 nan=0 time=654s
  Epoch 1 [1500/4000] loss=9.3430 nan=0 time=983s
  Epoch 1 [2000/4000] loss=9.1720 nan=0 time=1309s
  Epoch 1 [2500/4000] loss=9.0433 nan=0 time=1636s
  Epoch 1 [3000/4000] loss=8.9384 nan=0 time=1963s
  Epoch 1 [3500/4000] loss=8.8423 nan=0 time=2288s
  Epoch 1 [4000/4000] loss=8.7560 nan=0 time=2615s

Epoch 1/3 — loss: 8.7560 nan: 0 time: 2615s
  Saved (loss=8.7560)

  Epoch 2 [500/4000] loss=8.0282 nan=0 time=328s
  Epoch 2 [1000/4000] loss=7.9637 nan=0 time=655s
  Epoch 2 [1500/4000] loss=7.8930 nan=0 time=983s
  Epoch 2 [2000/4000] loss=7.8317 nan=0 time=1312s
  Epoch 2 [2500/4000] loss=7.7713 nan=0 time=1639s
  Epoch 2 [3000/4000] loss=7.7236 nan=0 time=1966s
  Epoch 2 [3500/4000] loss=7.6775 nan=0 time=2294s
  Epoch 2 [4000/4000] loss=7.6357 nan=0 time=2622s

Epoch 2/3 — loss: 7.

In [ ]:
# Quick 200-example eval to check if v2 is better than v1

peft_model.eval()

sample = hotpot_dev[:200]
em_oracle, f1_oracle = [], []
em_bm25, f1_bm25 = [], []

for ex in sample:
    q = ex["question"]
    gold = ex["answer"]

    # Oracle
    oracle_ctx = build_hotpot_supporting_context(ex)
    inputs = tokenizer(
        build_qa_prompt(q, oracle_ctx),
        return_tensors="pt", truncation=True, max_length=512
    ).to(device)
    with torch.no_grad():
        out = peft_model.generate(**inputs, max_new_tokens=32, num_beams=2, early_stopping=True)
    pred = tokenizer.decode(out[0], skip_special_tokens=True).strip()
    em_oracle.append(compute_em(pred, gold))
    f1_oracle.append(compute_f1(pred, gold))

    # BM25
    bm25_ctx, _, _ = build_hotpot_bm25_topk(ex, k=5)
    inputs = tokenizer(
        build_qa_prompt(q, bm25_ctx),
        return_tensors="pt", truncation=True, max_length=512
    ).to(device)
    with torch.no_grad():
        out = peft_model.generate(**inputs, max_new_tokens=32, num_beams=2, early_stopping=True)
    pred = tokenizer.decode(out[0], skip_special_tokens=True).strip()
    em_bm25.append(compute_em(pred, gold))
    f1_bm25.append(compute_f1(pred, gold))

print(f"Quick eval on {len(sample)} examples (v2 model):")
print(f"  M5 LoRA oracle: EM={sum(em_oracle)/len(em_oracle):.3f} F1={sum(f1_oracle)/len(f1_oracle):.3f}")
print(f"  M5 LoRA BM25:   EM={sum(em_bm25)/len(em_bm25):.3f} F1={sum(f1_bm25)/len(f1_bm25):.3f}")
print(f"\nv1 comparison (200 examples):")
print(f"  M5 LoRA oracle v1: EM=0.425, F1=0.611")
print(f"  M5 LoRA BM25 v1:   EM=0.070, F1=0.229")
print(f"\nZero-shot comparison:")
print(f"  B3 Oracle Large ZS: EM=0.453")
print(f"  B2 BM25 Large ZS:   EM=0.282")

Quick eval on 200 examples (v2 model):
  M5 LoRA oracle: EM=0.195 F1=0.342
  M5 LoRA BM25:   EM=0.170 F1=0.274

v1 comparison (200 examples):
  M5 LoRA oracle v1: EM=0.425, F1=0.611
  M5 LoRA BM25 v1:   EM=0.070, F1=0.229

Zero-shot comparison:
  B3 Oracle Large ZS: EM=0.453
  B2 BM25 Large ZS:   EM=0.282


In [ ]:
# Diagnose v2 outputs

for ex in hotpot_dev[:5]:
    oracle_ctx = build_hotpot_supporting_context(ex)
    inputs = tokenizer(
        build_qa_prompt(ex["question"], oracle_ctx),
        return_tensors="pt", truncation=True, max_length=512
    ).to(device)
    with torch.no_grad():
        out = peft_model.generate(**inputs, max_new_tokens=32, num_beams=2, early_stopping=True)
    pred = tokenizer.decode(out[0], skip_special_tokens=True).strip()

    print(f"Q: {ex['question'][:70]}")
    print(f"Gold: {ex['answer']}")
    print(f"Pred: '{pred}'")
    print(f"EM: {compute_em(pred, ex['answer'])}")
    print()

Q: Were Scott Derrickson and Ed Wood of the same nationality?
Gold: yes
Pred: 'American'
EM: 0

Q: What government position was held by the woman who portrayed Corliss A
Gold: Chief of Protocol
Pred: ''
EM: 0

Q: What science fantasy young adult series, told in first person, has a s
Gold: Animorphs
Pred: 'An'
EM: 0

Q: Are the Laleli Mosque and Esma Sultan Mansion located in the same neig
Gold: no
Pred: 'no'
EM: 1

Q: The director of the romantic comedy "Big Stone Gap" is based in what N
Gold: Greenwich Village, New York City
Pred: 'Green'
EM: 0



In [ ]:
# COMPLETE FRESH START: Setup + Dataset + Train

from google.colab import drive
drive.mount('/content/drive')
!pip -q install transformers accelerate sentencepiece rank_bm25 peft

import re, json, gc, time, random, torch
from pathlib import Path
from collections import Counter, defaultdict
from rank_bm25 import BM25Okapi
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model, TaskType

PROJECT_DIR = Path("/content/drive/MyDrive/421-Project")
device = "cuda"

# Load data
with open(PROJECT_DIR / "hotpot_dev_distractor_v1.json") as f:
    hotpot_dev = json.load(f)
with open(PROJECT_DIR / "hotpot_train_v1.1.json") as f:
    hotpot_train = json.load(f)
print(f"Dev: {len(hotpot_dev)} | Train: {len(hotpot_train)}")

# Helpers
def normalize_text(s):
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9\s]", "", s)
    return " ".join(s.split())

def compute_em(pred, gold):
    return int(normalize_text(pred) == normalize_text(gold))

def compute_f1(pred, gold):
    pt = normalize_text(pred).split()
    gt = normalize_text(gold).split()
    common = Counter(pt) & Counter(gt)
    ns = sum(common.values())
    if not pt or not gt: return int(pt == gt)
    if ns == 0: return 0.0
    p, r = ns/len(pt), ns/len(gt)
    return 2*p*r/(p+r)

def simple_tokenize(text):
    return re.findall(r"\w+", text.lower())

def get_qtype(ex):
    return ex.get("type", "unknown")

def hotpot_context_to_text(c):
    title, sents = c
    return f"Title: {title}\n{' '.join(sents)}"

def build_hotpot_all_context(ex):
    return "\n\n".join(hotpot_context_to_text(c) for c in ex["context"])

def build_hotpot_supporting_context(ex):
    supp = set(t for t, _ in ex["supporting_facts"])
    return "\n\n".join(hotpot_context_to_text(c) for c in ex["context"] if c[0] in supp)

def build_hotpot_bm25_topk(ex, k=5):
    para_texts = [hotpot_context_to_text(c) for c in ex["context"]]
    bm25 = BM25Okapi([simple_tokenize(t) for t in para_texts])
    scores = bm25.get_scores(simple_tokenize(ex["question"]))
    ranked = sorted(zip(ex["context"], para_texts, scores), key=lambda x: x[2], reverse=True)[:k]
    ctx = "\n\n".join(t for _, t, _ in ranked)
    supp = set(t for t, _ in ex["supporting_facts"])
    ret = set(c[0] for c, _, _ in ranked)
    return ctx, len(supp & ret)/len(supp) if supp else 0, int(supp <= ret)

def build_qa_prompt(question, context):
    return f"Answer the question using the context below.\n\nQuestion: {question}\n\nContext:\n{context}\n\nAnswer:"

# Load model
MODEL_NAME = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto")
print(f"Model loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# LoRA
lora_config = LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM, r=16, lora_alpha=32,
                         lora_dropout=0.05, target_modules=["q", "v"])
model.enable_input_require_grads()
model.gradient_checkpointing_enable()
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

# Dataset v3: mixed context + multi-task target (same as v1 format)
class HotpotMixedMultitask(Dataset):
    def __init__(self, data, tok, max_input_len=512, max_target_len=64, max_examples=5000):
        self.tok, self.max_input_len, self.max_target_len = tok, max_input_len, max_target_len
        self.examples = []
        random.seed(42)
        sample = random.sample(data, min(max_examples, len(data)))

        for ex in sample:
            q, ans = ex["question"], ex["answer"]
            supp_titles = set(t for t, _ in ex["supporting_facts"])
            oracle_paras = [c for c in ex["context"] if c[0] in supp_titles]
            oracle_ctx = "\n\n".join(hotpot_context_to_text(c) for c in oracle_paras)
            bm25_ctx, _, _ = build_hotpot_bm25_topk(ex, k=5)
            titles = ", ".join(c[0] for c in oracle_paras)

            target = f"Answer: {ans}\nSupporting: {titles}"
            prompt_tmpl = "Answer the question and identify supporting paragraphs.\n\nQuestion: {q}\n\nContext:\n{ctx}\n\nAnswer and supporting paragraphs:"

            # Oracle example
            self.examples.append((prompt_tmpl.format(q=q, ctx=oracle_ctx), target))
            # BM25 example
            self.examples.append((prompt_tmpl.format(q=q, ctx=bm25_ctx), target))

        random.shuffle(self.examples)
        print(f"Training examples: {len(self.examples)} (oracle + BM25 mixed)")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        inp, tgt = self.examples[idx]
        inputs = self.tok(inp, max_length=self.max_input_len, truncation=True,
                         padding="max_length", return_tensors="pt")
        targets = self.tok(tgt, max_length=self.max_target_len, truncation=True,
                          padding="max_length", return_tensors="pt")
        labels = targets["input_ids"].squeeze()
        labels[labels == self.tok.pad_token_id] = -100
        return {"input_ids": inputs["input_ids"].squeeze(),
                "attention_mask": inputs["attention_mask"].squeeze(),
                "labels": labels}

train_dataset = HotpotMixedMultitask(hotpot_train, tokenizer, max_examples=5000)

# Train
BATCH_SIZE, GRAD_ACCUM, EPOCHS, LR = 2, 8, 3, 5e-5
loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
opt = torch.optim.AdamW(peft_model.parameters(), lr=LR, weight_decay=0.01, eps=1e-4)
total_steps = len(loader) * EPOCHS // GRAD_ACCUM
warmup = total_steps // 10

peft_model.train()
step, best = 0, float("inf")
print(f"\nTraining: {len(train_dataset)} examples, eff_batch={BATCH_SIZE*GRAD_ACCUM}, steps={total_steps}")

for epoch in range(EPOCHS):
    eloss, esteps, nans = 0, 0, 0
    t0 = time.time()
    opt.zero_grad()
    for bi, batch in enumerate(loader):
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labs = batch["labels"].to(device)
        try:
            out = peft_model(input_ids=ids, attention_mask=mask, labels=labs)
            loss = out.loss / GRAD_ACCUM
            if torch.isnan(loss): nans += 1; opt.zero_grad(); continue
            loss.backward(); eloss += out.loss.item(); esteps += 1
        except RuntimeError:
            torch.cuda.empty_cache(); opt.zero_grad(); nans += 1; continue
        if (bi+1) % GRAD_ACCUM == 0:
            s = max(0.01, step/warmup) if step < warmup else max(0.1, 1-(step-warmup)/(total_steps-warmup))
            for pg in opt.param_groups: pg["lr"] = LR * s
            torch.nn.utils.clip_grad_norm_(peft_model.parameters(), 0.5)
            opt.step(); opt.zero_grad(); step += 1
        if (bi+1) % 500 == 0:
            print(f"  E{epoch+1} [{bi+1}/{len(loader)}] loss={eloss/max(esteps,1):.4f} nan={nans} t={time.time()-t0:.0f}s")
    avg = eloss/max(esteps,1)
    print(f"\nEpoch {epoch+1}/{EPOCHS} — loss: {avg:.4f} nan: {nans} time: {time.time()-t0:.0f}s")
    if 0 < avg < best:
        best = avg
        peft_model.save_pretrained("/content/drive/MyDrive/421-Project/hotpot_lora_v3_best")
        print(f"  Saved (loss={best:.4f})")
    print()

print(f"Done! Best loss: {best:.4f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dev: 7405 | Train: 90447


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded: 3.9 GB
trainable params: 4,718,592 || all params: 787,868,672 || trainable%: 0.5989
Training examples: 10000 (oracle + BM25 mixed)

Training: 10000 examples, eff_batch=16, steps=1875
  E1 [500/5000] loss=9.8163 nan=0 t=330s
  E1 [1000/5000] loss=9.8104 nan=0 t=660s
  E1 [1500/5000] loss=9.7802 nan=0 t=991s
  E1 [2000/5000] loss=9.6804 nan=0 t=1319s
  E1 [2500/5000] loss=9.5361 nan=0 t=1650s
  E1 [3000/5000] loss=9.4030 nan=0 t=1981s
  E1 [3500/5000] loss=9.2800 nan=0 t=2312s
  E1 [4000/5000] loss=9.1710 nan=0 t=2643s
  E1 [4500/5000] loss=9.0746 nan=0 t=2974s
  E1 [5000/5000] loss=8.9870 nan=0 t=3305s

Epoch 1/3 — loss: 8.9870 nan: 0 time: 3305s
  Saved (loss=8.9870)

  E2 [500/5000] loss=8.1076 nan=0 t=329s
  E2 [1000/5000] loss=8.0745 nan=0 t=659s
  E2 [1500/5000] loss=8.0392 nan=0 t=989s
  E2 [2000/5000] loss=8.0012 nan=0 t=1318s
  E2 [2500/5000] loss=7.9663 nan=0 t=1648s
  E2 [3000/5000] loss=7.9316 nan=0 t=1982s
  E2 [3500/5000] loss=7.8987 nan=0 t=2322s
  E2 [4000/5

In [ ]:
# Quick 200-example eval for v3

peft_model.eval()

def extract_m5_answer(output):
    import re
    match = re.search(r"Answer:\s*(.*?)(?:\n|Supporting:|$)", output, re.IGNORECASE)
    if match:
        ans = match.group(1).strip().rstrip(".")
        if ans.lower().startswith("yes"): return "yes"
        if ans.lower().startswith("no"): return "no"
        return ans
    lines = [l.strip() for l in output.strip().split('\n') if l.strip()]
    return lines[0] if lines else output.strip()

def build_m5_prompt(question, context):
    return (f"Answer the question and identify supporting paragraphs.\n\n"
            f"Question: {question}\n\nContext:\n{context}\n\n"
            f"Answer and supporting paragraphs:")

sample = hotpot_dev[:200]
em_oracle, f1_oracle = [], []
em_bm25, f1_bm25 = [], []

for ex in sample:
    q, gold = ex["question"], ex["answer"]

    # Oracle
    ctx = build_hotpot_supporting_context(ex)
    inputs = tokenizer(build_m5_prompt(q, ctx), return_tensors="pt",
                      truncation=True, max_length=512).to(device)
    with torch.no_grad():
        out = peft_model.generate(**inputs, max_new_tokens=64, num_beams=2, early_stopping=True)
    pred = extract_m5_answer(tokenizer.decode(out[0], skip_special_tokens=True))
    em_oracle.append(compute_em(pred, gold))
    f1_oracle.append(compute_f1(pred, gold))

    # BM25
    bm25_ctx, _, _ = build_hotpot_bm25_topk(ex, k=5)
    inputs = tokenizer(build_m5_prompt(q, bm25_ctx), return_tensors="pt",
                      truncation=True, max_length=512).to(device)
    with torch.no_grad():
        out = peft_model.generate(**inputs, max_new_tokens=64, num_beams=2, early_stopping=True)
    pred = extract_m5_answer(tokenizer.decode(out[0], skip_special_tokens=True))
    em_bm25.append(compute_em(pred, gold))
    f1_bm25.append(compute_f1(pred, gold))

print(f"v3 Quick eval (200 examples):")
print(f"  M5 LoRA oracle: EM={sum(em_oracle)/len(em_oracle):.3f} F1={sum(f1_oracle)/len(f1_oracle):.3f}")
print(f"  M5 LoRA BM25:   EM={sum(em_bm25)/len(em_bm25):.3f} F1={sum(f1_bm25)/len(f1_bm25):.3f}")
print(f"\nPrevious versions:")
print(f"  v1: oracle=0.425, BM25=0.070")
print(f"  v2: oracle=0.195, BM25=0.170")
print(f"\nZero-shot Large baselines:")
print(f"  B3 Oracle: EM=0.453")
print(f"  B2 BM25:   EM=0.282")

v3 Quick eval (200 examples):
  M5 LoRA oracle: EM=0.330 F1=0.549
  M5 LoRA BM25:   EM=0.235 F1=0.354

Previous versions:
  v1: oracle=0.425, BM25=0.070
  v2: oracle=0.195, BM25=0.170

Zero-shot Large baselines:
  B3 Oracle: EM=0.453
  B2 BM25:   EM=0.282


In [ ]:
# HotpotQA Method 5 LoRA v3 — Full eval on 2417 examples
# Paste this as a SINGLE CELL in Colab (A100 runtime)
# Estimated time: ~20-25 min on A100

# ── Setup ────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

!pip -q install transformers accelerate sentencepiece rank_bm25 peft
!pip -q install --upgrade torchao

import re, json, gc, time, torch
from pathlib import Path
from collections import Counter, defaultdict
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

PROJECT_DIR = Path("/content/drive/MyDrive/421-Project")
device = "cuda"

# ── Load data ────────────────────────────────────────────────────────────
with open(PROJECT_DIR / "hotpot_dev_distractor_v1.json") as f:
    hotpot_dev = json.load(f)
print(f"Full dev set: {len(hotpot_dev)}")

subset = hotpot_dev[:2417]   # Match MuSiQue eval size
print(f"Eval subset: {len(subset)}")

# ── Helpers ──────────────────────────────────────────────────────────────
def normalize_text(s):
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9\s]", "", s)
    return " ".join(s.split())

def compute_em(pred, gold):
    return int(normalize_text(pred) == normalize_text(gold))

def compute_f1(pred, gold):
    pt = normalize_text(pred).split()
    gt = normalize_text(gold).split()
    common = Counter(pt) & Counter(gt)
    ns = sum(common.values())
    if not pt or not gt: return int(pt == gt)
    if ns == 0: return 0.0
    p, r = ns / len(pt), ns / len(gt)
    return 2 * p * r / (p + r)

def simple_tokenize(text):
    return re.findall(r"\w+", text.lower())

def get_qtype(ex):
    return ex.get("type", "unknown")

def hotpot_context_to_text(c):
    title, sents = c
    return f"Title: {title}\n{' '.join(sents)}"

def build_hotpot_supporting_context(ex):
    supp = set(t for t, _ in ex["supporting_facts"])
    return "\n\n".join(hotpot_context_to_text(c) for c in ex["context"] if c[0] in supp)

def build_hotpot_bm25_topk(ex, k=5):
    para_texts = [hotpot_context_to_text(c) for c in ex["context"]]
    bm25 = BM25Okapi([simple_tokenize(t) for t in para_texts])
    scores = bm25.get_scores(simple_tokenize(ex["question"]))
    ranked = sorted(zip(ex["context"], para_texts, scores), key=lambda x: x[2], reverse=True)[:k]
    context = "\n\n".join(t for _, t, _ in ranked)
    supp = set(t for t, _ in ex["supporting_facts"])
    retrieved = set(c[0] for c, _, _ in ranked)
    recall = len(supp & retrieved) / len(supp) if supp else 0
    return context, recall, int(supp <= retrieved)

def build_m5_prompt(question, context):
    return (f"Answer the question and identify supporting paragraphs.\n\n"
            f"Question: {question}\n\nContext:\n{context}\n\n"
            f"Answer and supporting paragraphs:")

def extract_m5_answer(output):
    match = re.search(r"Answer:\s*(.*?)(?:\n|Supporting:|$)", output, re.IGNORECASE)
    if match:
        ans = match.group(1).strip().rstrip(".")
        if ans.lower().startswith("yes"): return "yes"
        if ans.lower().startswith("no"): return "no"
        return ans
    lines = [l.strip() for l in output.strip().split('\n') if l.strip()]
    return lines[0] if lines else output.strip()

# ── Load model + LoRA adapter ───────────────────────────────────────────
MODEL_NAME = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto"
)
print(f"Base model loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB")

peft_model = PeftModel.from_pretrained(
    base_model, str(PROJECT_DIR / "hotpot_lora_v3_best")
)
peft_model.eval()
print(f"LoRA v3 adapter loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# ── Evaluation ───────────────────────────────────────────────────────────
total = len(subset)

results = {
    "oracle": {"em": defaultdict(list), "f1": defaultdict(list)},
    "bm25":   {"em": defaultdict(list), "f1": defaultdict(list)},
}

start = time.time()

for i, ex in enumerate(subset, start=1):
    q, gold = ex["question"], ex["answer"]
    qtype = get_qtype(ex)

    # ── Oracle context ──
    oracle_ctx = build_hotpot_supporting_context(ex)
    inputs = tokenizer(
        build_m5_prompt(q, oracle_ctx),
        return_tensors="pt", truncation=True, max_length=512
    ).to(device)
    with torch.no_grad():
        out = peft_model.generate(
            **inputs, max_new_tokens=64, num_beams=2, early_stopping=True
        )
    pred = extract_m5_answer(tokenizer.decode(out[0], skip_special_tokens=True))
    results["oracle"]["em"][qtype].append(compute_em(pred, gold))
    results["oracle"]["f1"][qtype].append(compute_f1(pred, gold))
    results["oracle"]["em"]["all"].append(compute_em(pred, gold))
    results["oracle"]["f1"]["all"].append(compute_f1(pred, gold))

    # ── BM25 top-5 context ──
    bm25_ctx, _, _ = build_hotpot_bm25_topk(ex, k=5)
    inputs = tokenizer(
        build_m5_prompt(q, bm25_ctx),
        return_tensors="pt", truncation=True, max_length=512
    ).to(device)
    with torch.no_grad():
        out = peft_model.generate(
            **inputs, max_new_tokens=64, num_beams=2, early_stopping=True
        )
    pred = extract_m5_answer(tokenizer.decode(out[0], skip_special_tokens=True))
    results["bm25"]["em"][qtype].append(compute_em(pred, gold))
    results["bm25"]["f1"][qtype].append(compute_f1(pred, gold))
    results["bm25"]["em"]["all"].append(compute_em(pred, gold))
    results["bm25"]["f1"]["all"].append(compute_f1(pred, gold))

    # Progress
    if i % 200 == 0 or i == total:
        elapsed = time.time() - start
        o_em = sum(results["oracle"]["em"]["all"]) / len(results["oracle"]["em"]["all"])
        b_em = sum(results["bm25"]["em"]["all"]) / len(results["bm25"]["em"]["all"])
        remaining = (elapsed / i) * (total - i)
        print(f"  [{i}/{total}] Oracle EM={o_em:.3f}  BM25 EM={b_em:.3f}  "
              f"Time={elapsed:.0f}s  ~{remaining/60:.0f}min remaining")

# ── Final results ────────────────────────────────────────────────────────
elapsed = time.time() - start
print(f"\n{'='*90}")
print(f"HotpotQA Method 5 LoRA v3 — Full Eval ({total} examples, {elapsed/60:.1f} min)")
print(f"{'='*90}")

# By question type
for setting in ["oracle", "bm25"]:
    print(f"\n  {setting.upper()} context:")
    for qtype in sorted(set(results[setting]["em"].keys()) - {"all"}):
        em_vals = results[setting]["em"][qtype]
        f1_vals = results[setting]["f1"][qtype]
        print(f"    {qtype:>12s}:  EM={sum(em_vals)/len(em_vals):.3f}  "
              f"F1={sum(f1_vals)/len(f1_vals):.3f}  (n={len(em_vals)})")

    all_em = results[setting]["em"]["all"]
    all_f1 = results[setting]["f1"]["all"]
    print(f"    {'ALL':>12s}:  EM={sum(all_em)/len(all_em):.3f}  "
          f"F1={sum(all_f1)/len(all_f1):.3f}  (n={len(all_em)})")

# Comparison table
o_em = sum(results["oracle"]["em"]["all"]) / len(results["oracle"]["em"]["all"])
o_f1 = sum(results["oracle"]["f1"]["all"]) / len(results["oracle"]["f1"]["all"])
b_em = sum(results["bm25"]["em"]["all"]) / len(results["bm25"]["em"]["all"])
b_f1 = sum(results["bm25"]["f1"]["all"]) / len(results["bm25"]["f1"]["all"])

print(f"\n{'='*90}")
print(f"Cross-method comparison (n={total}):")
print(f"{'='*90}")
print(f"  {'Method':<35s} {'EM':>8s} {'F1':>8s}")
print(f"  {'-'*55}")
print(f"  {'M5 LoRA v3 (oracle)':<35s} {o_em:>8.3f} {o_f1:>8.3f}")
print(f"  {'M5 LoRA v3 (BM25 top-5)':<35s} {b_em:>8.3f} {b_f1:>8.3f}")
print(f"  {'-'*55}")
print(f"  {'ZS Large oracle (B3, full 7405)':<35s} {'0.453':>8s} {'—':>8s}")
print(f"  {'ZS Large BM25 (B2, full 7405)':<35s} {'0.282':>8s} {'—':>8s}")
print(f"  {'v3 quick eval (200 examples)':<35s} {'0.330':>8s} {'0.549':>8s}")
print(f"{'='*90}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 111.7 MB/s eta 0:00:00
Full dev set: 7405
Eval subset: 2417


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Base model loaded: 3.7 GB
LoRA v3 adapter loaded: 3.7 GB
  [200/2417] Oracle EM=0.330  BM25 EM=0.235  Time=195s  ~36min remaining
  [400/2417] Oracle EM=0.345  BM25 EM=0.207  Time=386s  ~32min remaining
  [600/2417] Oracle EM=0.337  BM25 EM=0.200  Time=582s  ~29min remaining
  [800/2417] Oracle EM=0.340  BM25 EM=0.209  Time=772s  ~26min remaining
  [1000/2417] Oracle EM=0.341  BM25 EM=0.217  Time=966s  ~23min remaining
  [1200/2417] Oracle EM=0.335  BM25 EM=0.217  Time=1161s  ~20min remaining
  [1400/2417] Oracle EM=0.336  BM25 EM=0.218  Time=1357s  ~16min remaining
  [1600/2417] Oracle EM=0.338  BM25 EM=0.213  Time=1549s  ~13min remaining
  [1800/2417] Oracle EM=0.332  BM25 EM=0.210  Time=1736s  ~10min remaining
  [2000/2417] Oracle EM=0.331  BM25 EM=0.213  Time=1931s  ~7min remaining
  [2200/2417] Oracle EM=0.330  BM25 EM=0.209  Time=2124s  ~3min remaining
  [2400/2417] Oracle EM=0.333  BM25 EM=0.211  Time=2311s  ~0min remaining
  [2417/2417] Oracle EM=0.333  BM25 EM=0.211  Time=2327